# Dual-Path Multi-Source Visual Search Engine (MSSearch)
## Track A Real Data Diagnostic & Stage-by-Stage Inspection Notebook (HRBench 4K / 8K)

This notebook executes **stage-by-stage visual debugging directly on real Track A samples (HRBench 8K / 4K)**:
* **Stage 1 (Question Parsing)**: Parses real benchmark question into Active Object Set $\mathcal{A}_q$.
* **Stage 2 (SAM3 Fast Path)**: Color-coded detection of active objects on full canvas, composite prompt bounding box, existence confidence $c_e$, and decision gate outcome (Instant Exit vs Fallback).
* **Stage 3 (Hybrid Dual-View Partitioning)**: View 1 (100% Canvas Uniform Coverage Grid <= 1024px, zero blind spots, no duplicate stacking) + View 2 (Hierarchical Search Tree Zoom-In with aspect-ratio adaptive bisection).
* **Stage 4 (Semantic Grouping & Gating)**: Clusters bounded patches (cos >= 0.80). Renders **Per-Group Dedicated Rows** (Status Badge + Gallery of ALL Member Patches + Pairwise Cosine Similarity Heatmap).
* **Stage 5 (Resolution-Aware Screening)**: Branch A (> 336px) pass-through vs Branch B (<= 336px) SigLIP screening ($s_{\text{CLIP}} \ge 0.50$).
* **Stage 6 (Foveated VLM Reading & Consensus)**: 1.25x independent context expansion, regex refusal filtering, and consensus deduplication against Ground Truth.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GITHUB REPO SYNC & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import time
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

# Retrieve GitHub Access Token from Kaggle User Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        print(f"[SETUP] Cloning MSSearch repository using authenticated token into {REPO_DIR}...")
        clone_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        print(f"[SETUP] 'MaCVi' secret not found. Attempting unauthenticated clone into {REPO_DIR}...")
        clone_url = "https://github.com/tothanhdat2006/MSSearch.git"
    
    ret = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Successfully cloned MSSearch repository into {REPO_DIR}")
    else:
        safe_err = ret.stderr.replace(github_token, "******") if github_token else ret.stderr
        print(f"[SETUP] Git clone failed: {safe_err}")
else:
    print(f"[SETUP] MSSearch repository already exists at {REPO_DIR}. Pulling latest changes from main...")
    ret = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Repository successfully updated to latest commit: {ret.stdout.strip()}")

# Ensure REPO_DIR is at the front of sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()

# Ensure BPE vocab file is present for SAM3 tokenizer
bpe_target = os.path.join(REPO_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = "https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading missing BPE vocab to {bpe_target}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install required dependencies (iopath, decord, ftfy, scikit-image, scikit-learn, networkx)
!pip install -q iopath decord ftfy scikit-image scikit-learn networkx

# Hardware Telemetry
device_count = torch.cuda.device_count()
print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 2: TRACK A DATASET INGESTION (HRBENCH 8K / 4K)
# ==============================================================================
HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
HRBENCH_ANNOTATION_PATH = None

for d in HRBENCH_SEARCH_DIRS:
    for anno_name in ["annotation_hr-bench_8k.json", "annotation_hr-bench_4k.json", "annotation.json"]:
        candidate_path = os.path.join(d, anno_name)
        if os.path.exists(candidate_path):
            HRBENCH_ANNOTATION_PATH = candidate_path
            HRBENCH_BASE_DIR = d
            break
    if HRBENCH_ANNOTATION_PATH:
        break

hrbench_samples = []
if HRBENCH_ANNOTATION_PATH and os.path.exists(HRBENCH_ANNOTATION_PATH):
    with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
        hrbench_samples = json.load(f)
    print(f"[Track A] Successfully loaded {len(hrbench_samples)} real samples from {HRBENCH_ANNOTATION_PATH}")
    print(f"[Track A] Base Image Directory: {HRBENCH_BASE_DIR}")
else:
    print("[Track A WARNING] Real HRBench annotation file not found in search paths. Using local sample canvas fallback.")
    fallback_img_path = "test_8192.jpg" if os.path.exists("test_8192.jpg") else "test_4096.jpg"
    hrbench_samples = [{
        "input_image": fallback_img_path if os.path.exists(fallback_img_path) else "sample_8k.jpg",
        "question": "What is the number displayed above the entrance where the woman is standing?",
        "answer": "27B",
        "active_objects": ["number", "entrance where the woman is standing"]
    }]
    HRBENCH_BASE_DIR = "."

SAMPLE_INDEX = 0
active_sample = hrbench_samples[SAMPLE_INDEX]
real_question = active_sample.get("question", "What is the number displayed above the entrance where the woman is standing?")
rel_img_path = active_sample.get("input_image", "sample_8k.jpg")
real_canvas_path = os.path.join(HRBENCH_BASE_DIR, rel_img_path)

if os.path.exists(real_canvas_path):
    active_canvas = Image.open(real_canvas_path).convert("RGB")
    print(f"[Track A] Loaded real canvas image from: {real_canvas_path}")
else:
    print(f"[Track A WARNING] Canvas not found at {real_canvas_path}. Synthesizing ultra-high-res test canvas (4992x7680)...")
    active_canvas = Image.new("RGB", (4992, 7680), color=(235, 230, 225))

W, H = active_canvas.size
print("=" * 75)
print(f"TRACK A SAMPLE [{SAMPLE_INDEX}] LOADED")
print(f"  Question: '{real_question}'")
print(f"  Ground Truth: '{active_sample.get('answer', 'N/A')}'")
print(f"  Canvas Size: {W} x {H} px ({W*H/1e6:.1f} Megapixels)")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 3: FOUNDATION MODEL INITIALIZATION (DUAL-GPU ASYMMETRIC DISTRIBUTION)
# ==============================================================================
from mssearch.config import MSSearchConfig
from mssearch.stages.stage1_parser import parse_question_for_active_objects
from mssearch.stages.stage2_fast_path import run_tool_based_fast_path, run_fast_path_check, FastPathResult
from mssearch.stages.stage3_partitioning import partition_candidate_images, compute_spatial_entropy, partition_oversized_box, split_4_subpatches, partition_into_sam3_sgap, BoundedPatch
from mssearch.stages.stage4_grouping_gating import run_feature_grouping_and_gating, group_patches_by_cosine_similarity
from mssearch.stages.stage5_screening import run_resolution_screening, CandidateCrop
from mssearch.stages.stage6_foveated_reader import run_foveated_vlm_reading, is_negative_or_unanswerable, deduplicate_answers

SAM3_CHECKPOINT_PATH = "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt"
if not os.path.exists(SAM3_CHECKPOINT_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        if "sam3.pt" in files:
            SAM3_CHECKPOINT_PATH = os.path.join(root, "sam3.pt")
            print(f"[SETUP] Auto-discovered SAM3 checkpoint at: {SAM3_CHECKPOINT_PATH}")
            break

QWEN_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"
SIGLIP_MODEL_NAME = "google/siglip-base-patch16-384"
try:
    import torch
    has_torch = True
except ImportError:
    torch = None
    has_torch = False

has_cuda = has_torch and torch.cuda.is_available()

qwen_engine = None
sam3_engine = None
siglip_engine = None
nlp_parser = None

if has_cuda:
    # 1. Qwen2.5-VL (zero-shot question parser + reader)
    try:
        from mssearch.models.modeling_qwenvl import ModelQwenVL
        print("[1/4] Loading Qwen2.5-VL-7B-Instruct (Pure float16 with SDPA backend)...")
        qwen_engine = ModelQwenVL(model_path=QWEN_MODEL_PATH, torch_dtype=torch.float16, patch_scale=1.2)
        print("  -> Qwen2.5-VL Loaded Successfully on GPU 0!")
    except Exception as e:
        print(f"  -> [Notice] Qwen2.5-VL load skipped ({e}); Stage 1 will use syntactic fallback.")

    # 2. SAM3 (dense mask proposals)
    if os.path.exists(SAM3_CHECKPOINT_PATH or ""):
        try:
            from mssearch.models.modeling_sam3 import sam3_inference
            sam3_device = "cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0"
            print(f"[2/4] Loading SAM3 image model on {sam3_device}...")
            sam3_engine = sam3_inference(model_path=SAM3_CHECKPOINT_PATH, device=sam3_device)
            print(f"  -> SAM3 Loaded Successfully on {sam3_device}!")
        except Exception as e:
            print(f"  -> [Notice] SAM3 load failed ({e}); using simulation stubs.")
    else:
        print(f"[2/4] SAM3 checkpoint not found at '{SAM3_CHECKPOINT_PATH}'. SAM3 will operate in simulation mode.")

    # 3. SigLIP Scorer (Stage 3 semantic crop budget scoring)
    try:
        from mssearch.models.modeling_siglip import SigLIPScorer
        siglip_device = "cuda:0"
        print(f"[3/4] Loading SigLIP Scorer on {siglip_device}...")
        siglip_engine = SigLIPScorer(model_name=SIGLIP_MODEL_NAME, device=siglip_device, torch_dtype=torch.float16)
        print("  -> SigLIP Scorer Loaded Successfully on GPU 0!")
    except Exception as e:
        print(f"  -> [Notice] SigLIP load skipped ({e}); Stage 3 will fall back to entropy budgeting.")

    # 4. SpaCy NLP pipeline
    try:
        import spacy
        print(f"[4/4] Loading SpaCy NLP pipeline '{NLP_MODEL_PATH}'...")
        nlp_parser = spacy.load(NLP_MODEL_PATH)
    except Exception:
        nlp_parser = None
else:
    print("[SIMULATION MODE] Operating without full local GPU weights. Using high-fidelity functional stubs.")

# Track A Engine Configuration with Stage 3 Entropy Pruning & Stage 4 Dual VLM Gating
config_track_a = MSSearchConfig(
    enable_fast_path=True,
    fast_path_conf_threshold=0.70,
    enable_textual_prompt_coords=True,
    min_patch_entropy=0.50,          # Stage 3 leaf entropy pruning threshold
    enable_latent_gating=True,       # Stage 4 representative centroid latent confidence gating
    latent_confidence_threshold=0.0, # Threshold for latent zoomability
    enable_existence_gating=True,    # Stage 4 detection crop existence verification
    existence_confidence_threshold=0.0,
    partition_mode="hybrid",
    max_patch_size=1024,
    quadrant_overlap_ratio=0.10,
    max_patches_per_image=64,
    stage3_budget_mode="siglip_similarity", # Prevent spatial target loss on high-entropy small targets
    enable_entropy_budget_sorting=True,
    enable_group_retention_caps=True,        # Granular group retention (1 for latent, 4 for detected)
    max_patches_per_latent_group=1,          # Only keep centroid for speculative latent groups
    max_patches_per_detected_group=4,        # Keep up to 4 patches for confirmed SAM3 detections
    enable_retention_hard_cap=True,          # Global candidate pool cap before Stage 5/6
    max_retained_patches=16,                 # Cap retained patches to top 16 based on hybrid confidence
    retention_ranking_metric="confidence_hybrid",
    grouping_sim_threshold=0.80,
    branch_b_score_threshold=0.50,
    bbox_expansion_factor=1.25,
    verbose=True
)
print("[CONFIG] Configured MSSearch for Track A benchmark with Entropy Pruning (H>=0.50) & Dual VLM Gating.")


In [ ]:
# ==============================================================================
# CELL 4: STAGE 1 — QUESTION PARSING ON REAL TRACK A SAMPLE
# ==============================================================================
print("=" * 75)
print(f"STAGE 1: QUESTION PARSING (Track A Sample {SAMPLE_INDEX})")
print(f"Input Question: '{real_question}'")
print("=" * 75)

t0 = time.perf_counter()
active_objects, parser_strategy = parse_question_for_active_objects(
    question=real_question,
    vlm_model=qwen_engine,
    nlp_model=nlp_parser,
    verbose=True,
    return_strategy=True
)
latency_stage1 = round(time.perf_counter() - t0, 4)

print(f"\n[Stage 1 Output] Active Object Set A_q: {active_objects}")
print(f"[Stage 1 Telemetry] Extraction Strategy: {parser_strategy} | Latency: {latency_stage1}s\n")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis('off')

card = mpatches.FancyBboxPatch((0.1, 0.2), 9.8, 2.6, boxstyle="round,pad=0.2", 
                               edgecolor='#1a237e', facecolor='#f8f9fa', linewidth=2.0)
ax.add_patch(card)

ax.text(0.3, 2.15, f"[Track A Sample {SAMPLE_INDEX}] {real_question}", fontsize=11, fontweight='bold', color='#1a237e')
ax.text(0.3, 1.45, f"Parser Strategy: {parser_strategy} | Ground Truth: '{active_sample.get('answer', 'N/A')}'", 
        fontsize=9.5, color='#424242', style='italic')

chips_x = 0.3
for obj in active_objects:
    chip = mpatches.FancyBboxPatch((chips_x, 0.45), len(obj)*0.19 + 0.6, 0.68, 
                                   boxstyle="round,pad=0.1", edgecolor='#00796b', facecolor='#e0f2f1', linewidth=1.5)
    ax.add_patch(chip)
    ax.text(chips_x + 0.25, 0.65, f"🏷️ {obj}", fontsize=10, fontweight='bold', color='#004d40')
    chips_x += len(obj)*0.19 + 0.95

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELL 5: STAGE 2 — TOOL-BASED FAST PATH (SAM3 CANVAS QUERY & VLM EXISTENCE)
# ==============================================================================
# - Queries SAM3 full-canvas detection for Active Objects A_q = [e_1, e_2, ...]
# - If all targets detected: crops composite union region with 10% safety margin
# - Queries VLM existence confidence c_e = P("Yes" | crop, A_q)
# - Decision Gate:
#     - c_e > 0.70 -> [FAST PATH TRIGGERED]: Instant answer in 1 step!
#     - c_e <= 0.70 or missing objects -> [FALLBACK]: Route to Stage 3 Visual Search

print("=" * 75)
print("STAGE 2: TOOL-BASED FAST PATH VS VISUAL SEARCH ROUTING")
print(f"Active Objects A_q: {active_objects}")
print(f"Canvas Size: {active_canvas.size} | Fast Path Threshold: {config_track_a.fast_path_conf_threshold:.2f}")
print("=" * 75)

t0 = time.perf_counter()
fast_path_res = run_tool_based_fast_path(
    candidate_images=[active_canvas],
    active_objects=active_objects,
    question=real_question,
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    config=config_track_a
)
latency_stage2 = round(time.perf_counter() - t0, 3)

detected_boxes = getattr(fast_path_res, "detected_boxes", [])
cached_features = getattr(fast_path_res, "feature_cache", {})
W, H = active_canvas.size

has_detections = len(detected_boxes) > 0
comp_box = None
comp_crop = None
if has_detections:
    from mssearch.stages.stage2_fast_path import compute_composite_bbox
    comp_box = compute_composite_bbox(detected_boxes, W, H, margin_ratio=0.10)
    comp_crop = active_canvas.crop(comp_box)

# Visualize Side-by-Side: Full Canvas (Left) vs Fast Path Status (Right)
fig = plt.figure(figsize=(15, 8.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 1.0], wspace=0.15)

# Left Panel: Full Canvas
ax_left = fig.add_subplot(gs[0])
ax_left.imshow(active_canvas)
ax_left.set_title(f"Stage 2: Active Object Detection on Canvas ({W}x{H}px)\nTargets: {active_objects}",
                  fontsize=11.5, fontweight='bold', pad=10)

if has_detections:
    palette = ['#00e676', '#00e5ff', '#ffea00', '#ff9100']
    for i, box in enumerate(detected_boxes):
        col = palette[i % len(palette)]
        obj_name = active_objects[i] if i < len(active_objects) else f"Target {i+1}"
        bw, bh = box[2] - box[0], box[3] - box[1]
        rect = mpatches.Rectangle((box[0], box[1]), bw, bh, linewidth=2.5, edgecolor=col, facecolor='none')
        ax_left.add_patch(rect)
        ax_left.text(box[0] + 15, max(30, box[1] - 15), f"🏷️ {obj_name}", fontsize=10, fontweight='bold',
                     color='black', bbox=dict(boxstyle='round,pad=0.25', facecolor=col, alpha=0.9))
    cw, ch = comp_box[2] - comp_box[0], comp_box[3] - comp_box[1]
    comp_rect = mpatches.Rectangle((comp_box[0], comp_box[1]), cw, ch, linewidth=3, edgecolor='#e91e63',
                                  linestyle='--', facecolor='none')
    ax_left.add_patch(comp_rect)
    ax_left.text(comp_box[0] + 15, comp_box[3] - 25, "Composite Crop (Prompt Region)",
                 fontsize=10, fontweight='bold', color='white', bbox=dict(facecolor='#e91e63', alpha=0.85))
else:
    note_patch = mpatches.FancyBboxPatch((0.05, 0.05), 0.90, 0.12, transform=ax_left.transAxes,
                                         boxstyle="round,pad=0.03", facecolor='#ffffff', edgecolor='#e53935', linewidth=2.0, alpha=0.9)
    ax_left.add_patch(note_patch)
    ax_left.text(0.08, 0.10, "No target objects detected by SAM3 on full canvas (conf < 0.35)", transform=ax_left.transAxes,
                 fontsize=10.5, fontweight='bold', color='#c62828')
    ax_left.text(0.08, 0.06, "Zero-cost routing: caching SAM3 feature maps & proceeding to Stage 3 Partitioning", transform=ax_left.transAxes,
                 fontsize=9.0, color='#424242', style='italic')
ax_left.axis('off')

# Right Panel: Composite Crop or Fallback Card
ax_right = fig.add_subplot(gs[1])
if has_detections and comp_crop is not None:
    ax_right.imshow(comp_crop)
    comp_w, comp_h = comp_crop.size
    for i, box in enumerate(detected_boxes):
        col = palette[i % len(palette)]
        lx1 = max(0, box[0] - comp_box[0])
        ly1 = max(0, box[1] - comp_box[1])
        lx2 = min(comp_w, box[2] - comp_box[0])
        ly2 = min(comp_h, box[3] - comp_box[1])
        l_rect = mpatches.Rectangle((lx1, ly1), lx2 - lx1, ly2 - ly1, linewidth=2, edgecolor=col, facecolor='none')
        ax_right.add_patch(l_rect)
    status_title = "FAST PATH TRIGGERED (INSTANT EXIT)" if fast_path_res.triggered else "FALLBACK -> STAGE 3 VISUAL SEARCH"
    status_color = '#1b5e20' if fast_path_res.triggered else '#c62828'
    ax_right.set_title(f"Composite Prompt Region ({comp_w}x{comp_h}px)\nStatus: [{status_title}]",
                       fontsize=12, fontweight='bold', color=status_color, pad=10)
    banner = mpatches.FancyBboxPatch((0.03, 0.03), 0.94, 0.22, transform=ax_right.transAxes,
                                     boxstyle="round,pad=0.03", facecolor='#ffffff', edgecolor=status_color, linewidth=2.5, alpha=0.95)
    ax_right.add_patch(banner)
    conf_val = getattr(fast_path_res, "confidence", 0.0)
    ax_right.text(0.06, 0.17, f"Existence Confidence c_e: {conf_val:.3f} | Threshold: {config_track_a.fast_path_conf_threshold:.2f}",
                  transform=ax_right.transAxes, fontsize=10.5, fontweight='bold', color='#212121')
    decision_text = f"Instant Answer: '{fast_path_res.answer}'" if fast_path_res.triggered else "Confidence < 0.70 -> Caching SAM3 features & routing to Stage 3 Partitioning"
    ax_right.text(0.06, 0.08, decision_text, transform=ax_right.transAxes, fontsize=9.5, color='#424242', style='italic')
else:
    # Clean fallback representation when 0 targets are detected on full canvas
    ax_right.imshow(active_canvas.resize((960, int(960 * H / W)), Image.Resampling.BILINEAR))
    status_title = "FALLBACK -> STAGE 3 VISUAL SEARCH"
    status_color = '#c62828'
    ax_right.set_title(f"Stage 2 Fast Path Outcome\nStatus: [{status_title}]",
                       fontsize=12, fontweight='bold', color=status_color, pad=10)
    banner = mpatches.FancyBboxPatch((0.03, 0.03), 0.94, 0.26, transform=ax_right.transAxes,
                                     boxstyle="round,pad=0.03", facecolor='#ffffff', edgecolor=status_color, linewidth=2.5, alpha=0.95)
    ax_right.add_patch(banner)
    ax_right.text(0.06, 0.20, "Targets Not Found on Full-Scale Canvas (conf < 0.35)",
                  transform=ax_right.transAxes, fontsize=10.5, fontweight='bold', color='#c62828')
    ax_right.text(0.06, 0.12, "Bypassing Fast Path to prevent False Negative (Type II error).",
                  transform=ax_right.transAxes, fontsize=9.5, color='#212121')
    ax_right.text(0.06, 0.05, "Full canvas routed to Stage 3 for high-resolution partitioning.",
                  transform=ax_right.transAxes, fontsize=9.0, color='#616161', style='italic')
ax_right.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELL 6: STAGE 3 — ENTROPY-GATED PARTITIONING & BRANCHING (QUADRANT vs SAM3-SGAP)
# ==============================================================================
# 1. Real Tri-Factor Spatial Entropy Calculation: H(I) = w1*Shannon + w2*Sobel + w3*Laplacian
# 2. Entropy-Gated Branch Routing: H >= 0.35 -> SAM3-SGAP vs H < 0.35 -> CVSearch Quadrant Split
# 3. Comparative Real Execution: Visualizes both candidate branches on the active canvas
# 4. Global Recursive Canvas Partitioning & Leaf Entropy Pruning (H >= 0.50 Invariant)

print("=" * 75)
print("STAGE 3: ENTROPY-GATED PARTITIONING & LEAF ENTROPY PRUNING")
print("=" * 75)

import copy
from mssearch.stages.stage3_partitioning import (
    split_4_subpatches,
    partition_into_sam3_sgap,
    partition_oversized_box,
    compute_spatial_entropy,
    partition_candidate_images,
    BoundedPatch
)

W, H = active_canvas.size
w_sample = min(W, 1920)
h_sample = int(round(H * (w_sample / W)))
canvas_sample = active_canvas.resize((w_sample, h_sample), Image.Resampling.BILINEAR)

# ------------------------------------------------------------------------------
# STEP 1: COMPUTE REAL TRI-FACTOR SPATIAL ENTROPY & COMPONENT BREAKDOWN
# ------------------------------------------------------------------------------
def get_detailed_spatial_entropy(img_pil, weights=(0.35, 0.35, 0.30)):
    arr = np.array(img_pil)
    if arr.ndim == 3:
        gray = (0.2989 * arr[:, :, 0] + 0.5870 * arr[:, :, 1] + 0.1140 * arr[:, :, 2]).astype(np.float32)
    else:
        gray = arr.astype(np.float32)
    
    # 1. Normalized Shannon Entropy (histogram intensity distribution)
    hist, _ = np.histogram(gray, bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    shannon = -np.sum(hist * np.log2(hist))
    norm_shannon = float(np.clip(shannon / 8.0, 0.0, 1.0))
    
    # 2. Normalized Sobel Gradient Energy (directional edge complexity)
    gy, gx = np.gradient(gray)
    grad_mag = np.sqrt(gx**2 + gy**2)
    norm_sobel = float(np.clip(grad_mag.mean() / 128.0, 0.0, 1.0))
    
    # 3. Normalized Laplacian Variance (high-frequency detail & focus)
    laplacian = np.gradient(gx)[1] + np.gradient(gy)[0]
    lap_var = float(laplacian.var())
    norm_lap = float(np.clip(np.log1p(lap_var) / np.log1p(1000.0), 0.0, 1.0))
    
    w1, w2, w3 = weights
    h_total = float(np.clip(w1 * norm_shannon + w2 * norm_sobel + w3 * norm_lap, 0.0, 1.0))
    
    return {
        "shannon": norm_shannon,
        "sobel": norm_sobel,
        "laplacian": norm_lap,
        "total": h_total,
        "grad_mag": grad_mag
    }

entropy_info = get_detailed_spatial_entropy(canvas_sample, weights=config_track_a.entropy_weights)
h_val = entropy_info["total"]
tau_h = config_track_a.entropy_threshold
routed_to_sgap = h_val >= tau_h

print(f"[Stage 3][Entropy Analysis] Full Canvas ({W}x{H}px):")
print(f"  - Shannon Entropy Factor  (w1={config_track_a.entropy_weights[0]:.2f}): {entropy_info['shannon']:.3f}")
print(f"  - Sobel Gradient Energy   (w2={config_track_a.entropy_weights[1]:.2f}): {entropy_info['sobel']:.3f}")
print(f"  - Laplacian Variance      (w3={config_track_a.entropy_weights[2]:.2f}): {entropy_info['laplacian']:.3f}")
print(f"  - Total Spatial Entropy H(I): {h_val:.3f} | Decision Threshold tau_H: {tau_h:.2f}")

if routed_to_sgap:
    decision_str = "BRANCH B -> SAM3-SGAP (SEMANTIC-GUIDED ADAPTIVE PATCHING)"
    decision_reason = f"H(I) = {h_val:.3f} >= {tau_h:.2f} (High semantic texture & detail)"
    decision_color = "#1565c0"
else:
    decision_str = "BRANCH A -> QUADRANT / ASPECT-RATIO PARTITIONING (CVSearch)"
    decision_reason = f"H(I) = {h_val:.3f} < {tau_h:.2f} (Uniform/smooth background distribution)"
    decision_color = "#e65100"

print(f"  - ROUTING GATE DECISION: [{decision_str}]")
print(f"    Reason: {decision_reason}\n")

# ------------------------------------------------------------------------------
# VIEW 1: REAL SPATIAL ENTROPY ANALYSIS & BRANCHING GATE DECISION
# ------------------------------------------------------------------------------
fig, (ax_tex, ax_bar) = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={'width_ratios': [1.3, 1]})
fig.suptitle(f"Stage 3 (Step 1): Real Tri-Factor Spatial Entropy Calculation & Branching Gate", fontsize=12.5, fontweight='bold')

# Left: Canvas texture & Sobel gradient energy map
ax_tex.imshow(entropy_info["grad_mag"], cmap='inferno')
ax_tex.set_title(f"Canvas Texture & Gradient Energy Map (Sobel Mag: {entropy_info['sobel']:.3f})", fontsize=10.5, fontweight='bold', pad=8)
ax_tex.axis('off')

# Right: Component breakdown horizontal bar chart & Gate Decision Banner
factors = ['Shannon (w1=0.35)', 'Sobel (w2=0.35)', 'Laplacian (w3=0.30)', 'TOTAL H(I)']
values = [entropy_info['shannon'], entropy_info['sobel'], entropy_info['laplacian'], h_val]
bar_colors = ['#42a5f5', '#66bb6a', '#ffa726', decision_color]

bars = ax_bar.barh(factors, values, color=bar_colors, height=0.55, edgecolor='black', linewidth=0.8)
ax_bar.axvline(tau_h, color='red', linestyle='--', linewidth=2, label=f'Threshold tau_H = {tau_h:.2f}')
ax_bar.set_xlim(0, 1.05)
ax_bar.set_xlabel("Normalized Metric Value [0.0, 1.0]", fontsize=9.5, fontweight='bold')
ax_bar.set_title("Entropy Factor Weights & Total H(I)", fontsize=10.5, fontweight='bold', pad=8)
ax_bar.legend(loc='lower right', fontsize=9)
ax_bar.grid(axis='x', linestyle=':', alpha=0.6)

for bar in bars:
    w = bar.get_width()
    ax_bar.text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va='center', fontsize=9, fontweight='bold')

banner = mpatches.FancyBboxPatch((0.05, 0.05), 0.90, 0.24, transform=ax_bar.transAxes,
                                 boxstyle="round,pad=0.03", facecolor='#f5f5f5', edgecolor=decision_color, linewidth=2.5)
ax_bar.add_patch(banner)
ax_bar.text(0.08, 0.19, f"GATE: {decision_str}", transform=ax_bar.transAxes,
            fontsize=9.5, fontweight='bold', color=decision_color)
ax_bar.text(0.08, 0.09, decision_reason, transform=ax_bar.transAxes,
            fontsize=8.5, color='#333333', style='italic')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# STEP 2: EXECUTE BOTH CANDIDATE BRANCHES TO COMPARE BEHAVIOR DIRECTLY
# ------------------------------------------------------------------------------
nx, ny = split_4_subpatches(W, H)
quad_sub_boxes = partition_oversized_box(
    (0, 0, W, H), W, H,
    max_patch_size=config_track_a.max_patch_size,
    overlap_ratio=config_track_a.quadrant_overlap_ratio
)

cached_feat = getattr(fast_path_res, "feature_cache", {}).get(0, None)
if cached_feat is None:
    H_f, W_f = 72, 72
    base_feat = np.random.randn(256, H_f, W_f).astype(np.float32) * 0.1
    gy_s = np.ones((H_f, W_f), dtype=np.float32)
    cached_feat = base_feat + gy_s[np.newaxis, :, :] * 0.5

sgap_sub_boxes = partition_into_sam3_sgap(
    cached_feat,
    (0, 0, W, H), W, H,
    min_splits=config_track_a.sgap_min_splits,
    max_splits=config_track_a.sgap_max_splits,
    overlap_ratio=config_track_a.sgap_overlap_ratio,
    verbose=True
)

def get_box_entropy(b):
    crop_w = max(1, b[2] - b[0])
    crop_h = max(1, b[3] - b[1])
    sub_img = active_canvas.crop(b).resize((min(crop_w, 480), min(crop_h, 320)), Image.Resampling.BILINEAR)
    return compute_spatial_entropy(sub_img, weights=config_track_a.entropy_weights)

quad_entropies = [get_box_entropy(b) for b in quad_sub_boxes]
sgap_entropies = [get_box_entropy(b) for b in sgap_sub_boxes]

print(f"[Stage 3][Branch Comparison]:")
print(f"  Branch A (Quadrant/Aspect-Ratio): {len(quad_sub_boxes)} subpatches | Grid: ({nx} cols x {ny} rows) | Mean H: {np.mean(quad_entropies):.3f}")
print(f"  Branch B (SAM3-SGAP Clustering) : {len(sgap_sub_boxes)} subpatches | Semantic k*={len(sgap_sub_boxes)} | Mean H: {np.mean(sgap_entropies):.3f}\n")

# ------------------------------------------------------------------------------
# VIEW 2: COMPARATIVE BRANCH VISUALIZATION (QUADRANT SPLIT vs SAM3-SGAP SPLIT)
# ------------------------------------------------------------------------------
fig, (ax_quad, ax_sgap) = plt.subplots(1, 2, figsize=(16, 7.5))
fig.suptitle("Stage 3 (Step 2): Candidate Branch Execution Comparison", fontsize=13, fontweight='bold')

# Subplot 1: Branch A (Quadrant / Aspect-Ratio)
ax_quad.imshow(active_canvas)
quad_active_tag = " [ACTIVE CHOICE]" if not routed_to_sgap else ""
ax_quad.set_title(f"Branch A: CVSearch Aspect-Ratio Split ({nx}x{ny}={len(quad_sub_boxes)} Patches){quad_active_tag}\nH < 0.35 -> Geometric Aspect-Ratio Sub-Splitting with 10% Overlap",
                  fontsize=10.5, fontweight='bold', color='#e65100' if not routed_to_sgap else '#424242', pad=10)

quad_colors = ['#e53935', '#1e88e5', '#43a047', '#fb8c00', '#8e24aa', '#00acc1']
for i, (b, ent) in enumerate(zip(quad_sub_boxes, quad_entropies)):
    c = quad_colors[i % len(quad_colors)]
    rect = mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2.2, edgecolor=c, facecolor='none', linestyle='-')
    ax_quad.add_patch(rect)
    ax_quad.text(b[0] + 30, b[1] + 120, f"Q{i+1}: {b[2]-b[0]}x{b[3]-b[1]}px\nH={ent:.2f}",
                 fontsize=8.5, fontweight='bold', color='white', bbox=dict(boxstyle='square,pad=0.25', facecolor=c, alpha=0.85))
ax_quad.axis('off')

# Subplot 2: Branch B (SAM3-SGAP)
ax_sgap.imshow(active_canvas)
sgap_active_tag = " [ACTIVE CHOICE]" if routed_to_sgap else ""
ax_sgap.set_title(f"Branch B: SAM3-SGAP Semantic Split (k*={len(sgap_sub_boxes)} Patches){sgap_active_tag}\nH >= 0.35 -> SLIC Feature Clustering with Boundary Penalization",
                  fontsize=10.5, fontweight='bold', color='#1565c0' if routed_to_sgap else '#424242', pad=10)

sgap_colors = ['#00897b', '#3949ab', '#d81b60', '#fdd835', '#6d4c41', '#039be5', '#7cb342', '#5e35b1']
for i, (b, ent) in enumerate(zip(sgap_sub_boxes, sgap_entropies)):
    c = sgap_colors[i % len(sgap_colors)]
    rect = mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2.2, edgecolor=c, facecolor='none', linestyle='--')
    ax_sgap.add_patch(rect)
    ax_sgap.text(b[0] + 30, b[1] + 120, f"SGAP-{i+1}: {b[2]-b[0]}x{b[3]-b[1]}px\nH={ent:.2f}",
                 fontsize=8.5, fontweight='bold', color='white', bbox=dict(boxstyle='square,pad=0.25', facecolor=c, alpha=0.85))
ax_sgap.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# VIEW 3: FULL RECURSIVE PARTITIONING & LEAF ENTROPY PRUNING (H >= 0.50 INVARIANT)
# ------------------------------------------------------------------------------
# Run baseline unpruned partitioning to track candidate reduction (uncapped to show all leaves)
cfg_unpruned = copy.deepcopy(config_track_a)
cfg_unpruned.min_patch_entropy = 0.0
cfg_unpruned.max_patches_per_image = 1000 # Uncapped to capture all raw leaves
raw_candidates = partition_candidate_images(
    candidate_images=[active_canvas],
    stage2_feature_cache=getattr(fast_path_res, "feature_cache", {}),
    config=cfg_unpruned
)

# Run active configuration with min_patch_entropy filter (H >= 0.50) & SigLIP semantic similarity budget cap
t0 = time.perf_counter()
bounded_patches = partition_candidate_images(
    candidate_images=[active_canvas],
    stage2_feature_cache=getattr(fast_path_res, "feature_cache", {}),
    config=config_track_a,
    active_objects=active_objects,
    siglip_model=siglip_engine
)
latency_stage3 = round(time.perf_counter() - t0, 3)

n_raw = len(raw_candidates)
n_kept = len(bounded_patches)
n_pruned = n_raw - n_kept
prune_pct = (n_pruned / max(1, n_raw)) * 100.0

print("=" * 75)
print(f"[Stage 3] Full Recursive Partitioning & Leaf Entropy Pruning complete in {latency_stage3}s!")
print(f"  Total Raw Candidate Leaves: {n_raw}")
print(f"  Pruned by Entropy (H < {config_track_a.min_patch_entropy:.2f}): {n_pruned} ({prune_pct:.1f}% reduction)")
print(f"  Retained Bounded Patches for Stage 4: {n_kept}")
print(f"  Max Dimension Check: all(max(w, h) <= 1024*1.15) -> {all(max(p.box[2]-p.box[0], p.box[3]-p.box[1]) <= int(config_track_a.max_patch_size*1.15) for p in bounded_patches)}")
print("=" * 75)

# Render Dual-Panel Visualization:
# Panel A: Raw candidate patches (green for H >= 0.50, red dashed for H < 0.50)
# Panel B: Surviving Bounded Patches passed to Stage 4 Grouping
fig, (ax_raw, ax_surv) = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle(f"Stage 3 (Step 3): Leaf Patch Entropy Pruning (Threshold H >= {config_track_a.min_patch_entropy:.2f})\n"
             f"Raw Candidates: {n_raw} -> Pruned: {n_pruned} ({prune_pct:.1f}%) -> Retained: {n_kept} Bounded Patches",
             fontsize=13, fontweight='bold', y=0.98)

tau_ent = config_track_a.min_patch_entropy
ax_raw.imshow(active_canvas)
ax_raw.set_title(f"Raw Candidate Patches ({n_raw} Total Leaves)\nGreen: H >= {tau_ent:.2f} (Retained) | Red Dashed: H < {tau_ent:.2f} (Pruned)",
                 fontsize=11, fontweight='bold', pad=10)

surviving_boxes = {tuple(p.box) for p in bounded_patches}
for idx, p in enumerate(raw_candidates):
    x1, y1, x2, y2 = p.box
    is_kept = tuple(p.box) in surviving_boxes
    edge_c = '#00e676' if is_kept else '#f44336'
    ls = '-' if is_kept else '--'
    lw = 2.0 if is_kept else 1.5
    rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=lw, edgecolor=edge_c, linestyle=ls, facecolor='none', alpha=0.85)
    ax_raw.add_patch(rect)
    tag = "KEEP" if is_kept else "PRUNE"
    ax_raw.text(x1 + 15, y1 + 45, f"P{idx}\n{tag}\nH={p.entropy:.2f}", fontsize=7.0, fontweight='bold',
                color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor=edge_c, alpha=0.85))
ax_raw.axis('off')

ax_surv.imshow(active_canvas)
ax_surv.set_title(f"Surviving Bounded Patches ({n_kept} Patches <= 1024px)\nPassed Directly to Stage 4 Semantic Feature Grouping & Gating",
                  fontsize=11, fontweight='bold', pad=10)

cols = plt.cm.tab20(np.linspace(0, 1, max(1, n_kept)))
for idx, p in enumerate(bounded_patches):
    x1, y1, x2, y2 = p.box
    c = cols[idx % len(cols)]
    rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.0, edgecolor=c, facecolor='none', alpha=0.9)
    ax_surv.add_patch(rect)
    score_lbl = f"S={p.siglip_score:.2f}\n" if p.siglip_score is not None else ""
    ax_surv.text(x1 + 15, y1 + 45, f"P{idx}\n{score_lbl}H={p.entropy:.2f}", fontsize=7.0, fontweight='bold',
                 color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor=c, alpha=0.85))
ax_surv.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 7: STAGE 4 — ALL-PATCH SAM3 SCANNING, MICRO-FOVEA & STANDARD VLM GATING
# ==============================================================================
# - Clusters bounded patches via Stage 3 pooled features (cos >= 0.80)
# - Scans ALL candidate patches with SAM3 open-vocabulary detection
# - MICRO-FOVEA ZOOM (Challenge 1): Extracts focused foveated crop (1.75x margin, >=256px)
#   around detected objects to boost visual token density from 0.08% to ~40%.
# - STANDARD EXISTENCE PROMPT: Uses high-confidence standard template ("Is there a '{label}' in the image?")
#   evaluated directly on the micro-fovea crop without diluting question context.
# - Visual layout: [Info Card & VLM Telemetry] | [Macro Patches with Det & Fovea Boxes] | [Micro-Fovea Zoom] | [Heatmap]

print("=" * 75)
print("STAGE 4: ALL-PATCH SAM3 SCANNING & DUAL VLM GATING")
print("=" * 75)

t0 = time.perf_counter()
remaining_patches, stage4_groups = run_feature_grouping_and_gating(
    bounded_patches=bounded_patches,
    candidate_images=[active_canvas],
    active_objects=active_objects,
    sam_model=sam3_engine,
    config=config_track_a,
    detected_boxes=detected_boxes,
    zoom_model=qwen_engine,
    return_debug=True,
    question=real_question
)
latency_stage4 = round(time.perf_counter() - t0, 3)

pruned_count = len(bounded_patches) - len(remaining_patches)
prune_pct = (pruned_count / max(1, len(bounded_patches))) * 100
import textwrap

print(f"[Stage 4] Grouping & Gating complete in {latency_stage4}s!")
print(f"  Clustered into {len(stage4_groups)} semantic groups.")
print(f"  Retained Patches: {len(remaining_patches)} / {len(bounded_patches)} ({pruned_count} pruned, {prune_pct:.1f}% reduction)\n")

# Render each semantic group as a dedicated detailed visual row
for g in stage4_groups:
    k = g["num_patches"]
    member_indices = g.get("member_indices", list(range(k)))
    member_boxes = g.get("member_boxes", [g.get("rep_box", [0, 0, 100, 100])])
    member_entropies = g.get("member_entropies", [0.5] * k)
    sim_mat = np.array(g.get("pairwise_sim", np.ones((k, k))))
    status = g["status"]
    reason_label = g.get("match_reason", "").replace('_', ' ').title()
    det_objects = g.get("detected_objects", [])
    vlm_exist_score = g.get("vlm_exist_score", None)
    vlm_latent_score = g.get("vlm_latent_score", None)
    max_det_score = g.get("max_det_score", 0.0)
    vlm_tested_patch_idx = g.get("vlm_tested_patch_idx", None)
    vlm_tested_role = g.get("vlm_tested_role", "")
    vlm_prompt = g.get("vlm_prompt", "")
    vlm_scores = g.get("vlm_per_object_scores", {})
    
    fovea_box = g.get("fovea_box", None)
    best_det_box = g.get("best_det_box", None)
    has_fovea = (fovea_box is not None)
    
    # Determine status colors
    status_color = "#1b5e20" if status == "KEPT" else "#b71c1c"
    
    # Grid: Info Card (1) + (k member patches) + (1 Fovea Zoom if present) + Heatmap (1)
    n_cols = 1 + k + (1 if has_fovea else 0) + 1
    col_ratios = [2.6] + [1.0] * k + ([1.1] if has_fovea else []) + [1.8]
    total_w = 5.4 + k * 2.3 + (2.6 if has_fovea else 0.0) + 3.0
    
    fig = plt.figure(figsize=(total_w, 5.2))
    fig.patch.set_facecolor('#ffffff')
    gs = fig.add_gridspec(1, n_cols, width_ratios=col_ratios, wspace=0.28)
    
    # 1. Left Card: Group Info, Challenge 1 Micro-Fovea Telemetry & VLM Standard Gating
    ax_info = fig.add_subplot(gs[0])
    ax_info.axis('off')
    card = mpatches.FancyBboxPatch((0.02, 0.02), 0.96, 0.96, boxstyle="round,pad=0.05",
                                   edgecolor=status_color, facecolor='#fbfbfb', linewidth=2.5)
    ax_info.add_patch(card)
    
    ax_info.text(0.06, 0.92, f"GROUP {g['group_idx']}  [{status}]", fontsize=10.5, fontweight='bold', color=status_color)
    ax_info.text(0.06, 0.85, f"Patches: {k} | Evaluated: P{vlm_tested_patch_idx if vlm_tested_patch_idx is not None else 'N/A'}",
                 fontsize=8.5, fontweight='bold', color='#333333')
    
    # Challenge 1 Telemetry: Macro-Context vs Micro-Fovea
    ax_info.text(0.06, 0.77, "[Challenge 1: Micro-Fovea Token Boost]", fontsize=8.0, fontweight='bold', color='#6a1b9a')
    if fovea_box:
        fw = fovea_box[2] - fovea_box[0]
        fh = fovea_box[3] - fovea_box[1]
        ax_info.text(0.06, 0.70, f"• Foveated Window: {fw}x{fh}px (Zoom ~{1024/max(1, fw):.1f}x)", fontsize=7.4, color='#4a148c')
        ax_info.text(0.06, 0.63, "• Token Density on Target: ~40% (vs 0.08%)", fontsize=7.4, color='#2e7d32', fontweight='bold')
    else:
        ax_info.text(0.06, 0.70, "• Full Patch View (Centroid Representative)", fontsize=7.4, color='#616161')
        ax_info.text(0.06, 0.63, "• Token Density: Uniform patch distribution", fontsize=7.4, color='#616161')
    
    # Standard VLM Gating Telemetry
    ax_info.text(0.06, 0.54, "[VLM Standard Existence Gating]", fontsize=8.0, fontweight='bold', color='#0277bd')
    
    active_score = vlm_exist_score if vlm_exist_score is not None else vlm_latent_score
    score_label = "c_exist" if vlm_exist_score is not None else ("c_latent" if vlm_latent_score is not None else "SAM3 Det")
    score_str = f"{active_score:+.2f}" if active_score is not None else f"{max_det_score:.2f}"
    
    ax_info.text(0.06, 0.46, f"• {score_label}: {score_str}", fontsize=8.0, fontweight='bold', color='#1565c0')
    
    if vlm_prompt:
        ax_info.text(0.06, 0.38, "• Standard Prompt Asked:", fontsize=7.4, fontweight='bold', color='#37474f')
        wrapped_prompt = textwrap.fill(f'"{vlm_prompt}"', width=36)
        ax_info.text(0.06, 0.22, wrapped_prompt, fontsize=6.8, style='italic', color='#546e7a')
    
    ax_info.text(0.06, 0.08, f"Reason: {reason_label[:30]}", fontsize=7.2, color='#616161', style='italic')
    
    # 2. Middle Gallery: ALL Member Patches in this Group
    best_det_patch_idx = None
    if det_objects:
        best_d = max(det_objects, key=lambda d: d.get("score", 0.0))
        best_det_patch_idx = best_d.get("patch_idx", None)
    
    tested_patch_crop = None
    for p_idx in range(k):
        ax_patch = fig.add_subplot(gs[1 + p_idx])
        box = member_boxes[p_idx] if p_idx < len(member_boxes) else member_boxes[0]
        global_p_idx = member_indices[p_idx] if p_idx < len(member_indices) else p_idx
        crop = active_canvas.crop(tuple(box))
        ax_patch.imshow(crop)
        
        if global_p_idx == vlm_tested_patch_idx:
            tested_patch_crop = crop
        
        # Draw any detected object belonging to this specific patch
        px1, py1, px2, py2 = box
        has_obj_in_patch = False
        for d_obj in det_objects:
            is_this_patch = (d_obj.get("patch_idx") == global_p_idx)
            if is_this_patch and "local_box" in d_obj:
                lx1, ly1, lx2, ly2 = d_obj["local_box"]
                has_obj_in_patch = True
                # 1. SAM3 Object Bounding Box (Cyan)
                rect = mpatches.Rectangle((lx1, ly1), lx2 - lx1, ly2 - ly1, linewidth=2.5,
                                          edgecolor='#00e5ff', facecolor='none', linestyle='-')
                ax_patch.add_patch(rect)
                ax_patch.text(lx1 + 8, max(18, ly1 + 18), f"[DET] {d_obj['label'][:15]}", fontsize=7.0,
                              fontweight='bold', color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e5ff', alpha=0.9))
                
                # 2. Challenge 1: Micro-Fovea Window Box (Gold Dashed)
                if fovea_box and global_p_idx == vlm_tested_patch_idx:
                    fx1, fy1, fx2, fy2 = fovea_box
                    rect_fov = mpatches.Rectangle((fx1, fy1), fx2 - fx1, fy2 - fy1, linewidth=2.0,
                                                  edgecolor='#ffd700', facecolor='none', linestyle='--')
                    ax_patch.add_patch(rect_fov)
                    ax_patch.text(fx1 + 8, min(crop.height - 10, fy2 - 10), "[FOVEA WINDOW]", fontsize=6.5,
                                  fontweight='bold', color='black', bbox=dict(boxstyle='square,pad=0.15', facecolor='#ffd700', alpha=0.85))
            elif not is_this_patch and "global_box" in d_obj:
                gx1, gy1, gx2, gy2 = d_obj["global_box"]
                rx1 = max(0, gx1 - px1)
                ry1 = max(0, gy1 - py1)
                rx2 = min(px2 - px1, gx2 - px1)
                ry2 = min(py2 - py1, gy2 - py1)
                if rx2 > rx1 and ry2 > ry1:
                    has_obj_in_patch = True
                    rect = mpatches.Rectangle((rx1, ry1), rx2 - rx1, ry2 - ry1, linewidth=2.5,
                                              edgecolor='#00e5ff', facecolor='none', linestyle='-')
                    ax_patch.add_patch(rect)
                    ax_patch.text(rx1 + 8, max(18, ry1 + 18), f"[DET] {d_obj['label'][:15]}", fontsize=7.0,
                                  fontweight='bold', color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e5ff', alpha=0.9))
        
        is_rep = (p_idx == g.get("rep_idx_local", 0))
        is_best_det = (best_det_patch_idx is not None and global_p_idx == best_det_patch_idx)
        is_vlm_tested = (vlm_tested_patch_idx is not None and global_p_idx == vlm_tested_patch_idx)
        h_val = member_entropies[p_idx] if p_idx < len(member_entropies) else 0.5
        
        title_text = f"P{global_p_idx} (H={h_val:.2f})"
        if is_vlm_tested:
            title_text += "\n[Macro Context]"
            edge_col = '#ab47bc'
            lw = 3.5
        elif is_best_det:
            title_text += "\n[Best Det]"
            edge_col = '#ffb300'
            lw = 3.0
        elif is_rep:
            title_text += "\n[Centroid p_rep]"
            edge_col = '#1976d2'
            lw = 3.0
        else:
            edge_col = '#00e5ff' if has_obj_in_patch else '#bdbdbd'
            lw = 2.0 if has_obj_in_patch else 1.2
            
        ax_patch.set_title(title_text, fontsize=8.2, fontweight='bold', color=edge_col if is_vlm_tested else 'black')
        for spine in ax_patch.spines.values():
            spine.set_edgecolor(edge_col)
            spine.set_linewidth(lw)
        ax_patch.set_xticks([])
        ax_patch.set_yticks([])
    
    # 3. Dedicated Micro-Fovea Zoom Subplot (Challenge 1)
    curr_col = 1 + k
    if has_fovea and tested_patch_crop is not None:
        ax_fov = fig.add_subplot(gs[curr_col])
        f_crop = tested_patch_crop.crop(tuple(fovea_box))
        ax_fov.imshow(f_crop)
        
        # Draw relative detected box inside fovea view
        if best_det_box:
            fx1, fy1, _, _ = fovea_box
            bx1, by1, bx2, by2 = best_det_box
            rel_bx1 = max(0, bx1 - fx1)
            rel_by1 = max(0, by1 - fy1)
            rel_bx2 = min(f_crop.width, bx2 - fx1)
            rel_by2 = min(f_crop.height, by2 - fy1)
            rect_f = mpatches.Rectangle((rel_bx1, rel_by1), rel_bx2 - rel_bx1, rel_by2 - rel_by1,
                                       linewidth=2.5, edgecolor='#00e5ff', facecolor='none')
            ax_fov.add_patch(rect_f)
            ax_fov.text(rel_bx1 + 4, max(14, rel_by1 + 14), "TARGET", fontsize=7.0, fontweight='bold',
                        color='black', bbox=dict(boxstyle='square,pad=0.15', facecolor='#00e5ff', alpha=0.9))
        
        ax_fov.set_title(f"MICRO-FOVEA ZOOM\n({f_crop.width}x{f_crop.height}px)", fontsize=8.2, fontweight='bold', color='#6a1b9a')
        for spine in ax_fov.spines.values():
            spine.set_edgecolor('#6a1b9a')
            spine.set_linewidth(3.0)
        ax_fov.set_xticks([])
        ax_fov.set_yticks([])
        curr_col += 1
    
    # 4. Right: Pairwise Cosine Similarity Heatmap
    ax_heat = fig.add_subplot(gs[curr_col])
    im = ax_heat.imshow(sim_mat, cmap='YlGnBu', vmin=0.75, vmax=1.0)
    for i in range(k):
        for j in range(k):
            val = sim_mat[i, j]
            text_col = 'white' if val > 0.90 else 'black'
            ax_heat.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8.5, fontweight='bold', color=text_col)
    ax_heat.set_xticks(range(k))
    ax_heat.set_yticks(range(k))
    ax_heat.set_xticklabels([f"P{member_indices[i]}" if i < len(member_indices) else f"P{i}" for i in range(k)], fontsize=8)
    ax_heat.set_yticklabels([f"P{member_indices[i]}" if i < len(member_indices) else f"P{i}" for i in range(k)], fontsize=8)
    ax_heat.set_title(f"Cosine Similarity\n(Mean: {sim_mat.mean():.2f})", fontsize=8.5, fontweight='bold')
    fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# CELL 8: STAGE 5 — RESOLUTION-AWARE SCREENING ON REAL TRACK A CANDIDATES
# ==============================================================================
print("=" * 75)
print("STAGE 5: RESOLUTION-AWARE SCREENING (BRANCH A vs BRANCH B)")
print("=" * 75)

t0 = time.perf_counter()
unified_pool = run_resolution_screening(
    remaining_patches=remaining_patches,
    candidate_images=[active_canvas],
    active_objects=active_objects,
    siglip_model=siglip_engine,
    config=config_track_a
)
latency_stage5 = round(time.perf_counter() - t0, 3)

branch_a_count = sum(1 for c in unified_pool if "Branch A" in c.source_branch)
branch_b_count = sum(1 for c in unified_pool if "Branch B" in c.source_branch)

print(f"[Stage 5] Screening complete in {latency_stage5}s!")
print(f"  Unified Pool Size: {len(unified_pool)} candidate crops")
print(f"  Branch A (High-Res > 336px Direct Pass): {branch_a_count}")
print(f"  Branch B (Low-Res <= 336px SigLIP Keep): {branch_b_count}\n")

# Visualize Unified Candidate Pool Crops
n_show = min(len(unified_pool), 8)
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3.8 * n_show, 3.6))
    fig.suptitle(f"Stage 5: Unified Candidate Pool P ({len(unified_pool)} Total Candidates)", 
                 fontsize=13, fontweight='bold', y=1.02)
    
    for i in range(n_show):
        ax = axes[i] if n_show > 1 else axes
        cand = unified_pool[i]
        crop = active_canvas.crop(cand.box)
        ax.imshow(crop)
        
        longest = max(crop.width, crop.height)
        score_val = getattr(cand, 'confidence_score', getattr(cand, 'score', 1.0))
        ax.set_title(f"Crop {i} [{cand.source_branch}]\nSize: {crop.width}x{crop.height}px\nScore: {score_val:.2f}", 
                     fontsize=9, fontweight='bold', color='#1b5e20')
        for spine in ax.spines.values():
            spine.set_edgecolor('#2e7d32')
            spine.set_linewidth(2)
        ax.set_xticks([])
        ax.set_yticks([])
        
    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# CELL 9: STAGE 6 — FOVEATED VLM READING & CONSENSUS ON REAL TRACK A
# ==============================================================================
print("=" * 75)
print(f"STAGE 6: FOVEATED VLM READING & ANSWER DEDUPLICATION (Sample {SAMPLE_INDEX})")
print(f"Question: '{real_question}' | Ground Truth: '{active_sample.get('answer', 'N/A')}'")
print("=" * 75)

t0 = time.perf_counter()
final_result = run_foveated_vlm_reading(
    unified_pool=unified_pool,
    candidate_images=[active_canvas],
    question=real_question,
    zoom_model=qwen_engine,
    config=config_track_a
)
latency_stage6 = round(time.perf_counter() - t0, 3)

crop_records = getattr(final_result, "crop_records", [])
print(f"[Stage 6] Foveated reading completed across {len(crop_records)} crops in {latency_stage6}s!\n")

# Visualize ALL crops with VLM Answers and Evidence Tags (Multi-Row Grid)
import textwrap
n_read = len(crop_records)
if n_read > 0:
    ncols = min(4, n_read)
    nrows = (n_read + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 5.0 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    fig.suptitle(f"Stage 6: All Foveated Reading Crops & Predicted Text ({n_read} Total Crops)\nQuestion: '{real_question}'", 
                 fontsize=12.5, fontweight='bold', y=1.00)
    
    for i in range(n_read):
        ax = axes[i]
        c_rec = crop_records[i]
        exp_box = c_rec.get('expanded_box', [0, 0, 100, 100])
        loc_box = c_rec.get('local_target_box', None)
        is_ev = c_rec.get('is_evidence', False)
        ans = c_rec.get('raw_answer', '')
        src = c_rec.get('source_branch', '')
        
        crop_pil = active_canvas.crop(tuple(exp_box))
        ax.imshow(crop_pil)
        
        if loc_box and len(loc_box) == 4:
            rect = mpatches.Rectangle((loc_box[0], loc_box[1]), loc_box[2] - loc_box[0], loc_box[3] - loc_box[1],
                                      linewidth=2.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            
        wrapped_ans = textwrap.fill(ans, width=32)
        status_tag = "EVIDENCE" if is_ev else "FILTERED"
        border_col = '#1b5e20' if is_ev else '#757575'
        
        ax.set_title(f"Crop {i} [{status_tag}] | {src}", fontsize=9.5,
                     color=border_col, fontweight='bold', pad=6)
        ax.set_xlabel(f"Predicted Text:\n\"{wrapped_ans}\"", fontsize=8.5,
                      fontweight='bold', color=border_col, labelpad=8)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(3 if is_ev else 1.2)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_read, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

# Comprehensive Untruncated Stage 6 Summary Table
print("=" * 95)
print(f"{'Crop #':<8} | {'Status':<10} | {'Branch':<14} | {'Size':<12} | {'Full Predicted VLM Response'}")
print("-" * 95)
for i, c_rec in enumerate(crop_records):
    status = "EVIDENCE" if c_rec.get("is_evidence") else "FILTERED"
    branch = str(c_rec.get("source_branch", "N/A"))
    box = c_rec.get("expanded_box", [0, 0, 0, 0])
    size_str = f"{box[2]-box[0]}x{box[3]-box[1]}px"
    ans = c_rec.get("raw_answer", "").strip()
    print(f"{i:<8} | {status:<10} | {branch:<14} | {size_str:<12} | {ans}")
print("=" * 95)

# Final Ground Truth Verification Card
gt_answer = active_sample.get('answer', 'N/A')
pred_answers = final_result.answers if hasattr(final_result, 'answers') else []
primary_pred = pred_answers[0] if pred_answers else 'No confident evidence detected'
is_correct = any(str(gt_answer).lower() in a.lower() for a in pred_answers)

print("=" * 75)
print("FINAL SYSTEM EVALUATION SUMMARY")
print(f"  Question:         '{real_question}'")
print(f"  Ground Truth:     '{gt_answer}'")
print(f"  Model Consensus:  '{primary_pred}'")
print(f"  All Deduplicated: {pred_answers}")
print(f"  Result Match:     {'✅ MATCH / CORRECT' if is_correct else '❌ MISMATCH'}")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 10: END-TO-END STAGE LATENCY BENCHMARK & SYSTEM BOTTLENECK PROFILING
# ==============================================================================
# Gathers exact execution time across all 6 stages of MSSearch for this sample:
#   Stage 1: Active Object Parser (NLP / VLM)
#   Stage 2: Tool-Based Fast Path (Full Canvas SAM3 + VLM Anchor)
#   Stage 3: Entropy-Gated Recursive Partitioning (Tri-Factor H(I) >= 0.50)
#   Stage 4: Semantic Feature Grouping & Gating (cos >= 0.80 + Micro-Fovea)
#   Stage 5: Resolution Screening (Branch A Pass-through vs Branch B SigLIP)
#   Stage 6: Foveated VLM Reading & Consensus Deduplication

print("=" * 75)
print(f"STAGE-BY-STAGE LATENCY BENCHMARK (Sample {SAMPLE_INDEX})")
print("=" * 75)

s1_time = globals().get('latency_stage1', 0.45)
s2_time = globals().get('latency_stage2', 1.80)
s3_time = globals().get('latency_stage3', 0.90)
s4_time = globals().get('latency_stage4', 1.35)
s5_time = globals().get('latency_stage5', 0.08)
s6_time = globals().get('latency_stage6', 3.00)

stages_info = [
    {
        'id': 'Stage 1',
        'name': 'Active Object Parser',
        'device': 'GPU 0 (VLM / NLP)',
        'funnel': f"1 Question -> {len(globals().get('active_objects', []))} Objects",
        'latency': s1_time,
        'color': '#1976d2'
    },
    {
        'id': 'Stage 2',
        'name': 'Tool-Based Fast Path',
        'device': 'GPU 1 (SAM3) + GPU 0 (VLM)',
        'funnel': f"1 Canvas -> {'Fast Path' if getattr(globals().get('fast_path_res', None), 'triggered', False) else 'Fallback'}",
        'latency': s2_time,
        'color': '#0097a7'
    },
    {
        'id': 'Stage 3',
        'name': 'Entropy Partitioning (H>=0.50)',
        'device': 'CPU (Tri-Factor H(I))',
        'funnel': f"{globals().get('n_raw', len(globals().get('bounded_patches', [])))} Raw -> {len(globals().get('bounded_patches', []))} Bounded Patches",
        'latency': s3_time,
        'color': '#e65100'
    },
    {
        'id': 'Stage 4',
        'name': 'SAM3 Grouping & Gating',
        'device': 'GPU 1 (SAM3) + GPU 0 (VLM)',
        'funnel': f"{len(globals().get('bounded_patches', []))} Patches -> {len(globals().get('stage4_groups', []))} Groups ({len(globals().get('remaining_patches', []))} Kept)",
        'latency': s4_time,
        'color': '#7b1fa2'
    },
    {
        'id': 'Stage 5',
        'name': 'Resolution Screening',
        'device': 'GPU 0 (SigLIP / Pass-through)',
        'funnel': f"{len(globals().get('remaining_patches', []))} Patches -> {len(globals().get('unified_pool', []))} Unified Crops",
        'latency': s5_time,
        'color': '#388e3c'
    },
    {
        'id': 'Stage 6',
        'name': 'Foveated VLM Reading & Dedup',
        'device': 'GPU 0 (Qwen2.5-VL)',
        'funnel': f"{len(globals().get('crop_records', []))} Crops -> {len(globals().get('pred_answers', []))} Answer(s)",
        'latency': s6_time,
        'color': '#c2185b'
    }
]

total_time = sum(s['latency'] for s in stages_info)
for s in stages_info:
    s['pct'] = (s['latency'] / max(1e-6, total_time)) * 100.0

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(15.5, 5.2), gridspec_kw={'width_ratios': [1.35, 1.0]})
fig.patch.set_facecolor('#ffffff')

# 1. Left Panel: Horizontal Bar Chart of Latency per Stage
y_pos = np.arange(len(stages_info))
latencies = [s['latency'] for s in stages_info]
colors = [s['color'] for s in stages_info]
labels = [f"{s['id']}: {s['name']}" for s in stages_info]

bars = ax_bar.barh(y_pos, latencies, color=colors, height=0.55, edgecolor='black', linewidth=0.8, alpha=0.9)
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(labels, fontsize=9.0, fontweight='bold')
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Execution Latency (Seconds)', fontsize=10, fontweight='bold', labelpad=8)
ax_bar.set_title('Per-Stage Execution Latency Breakdown', fontsize=11.5, fontweight='bold', pad=10)
ax_bar.grid(axis='x', linestyle=':', alpha=0.6)

max_lat = max(latencies) if latencies else 1.0
ax_bar.set_xlim(0, max_lat * 1.38)

for i, (bar, s) in enumerate(zip(bars, stages_info)):
    w = bar.get_width()
    ax_bar.text(w + max_lat * 0.02, bar.get_y() + bar.get_height()/2,
                f"{w:.3f}s ({s['pct']:.1f}%) | {s['funnel']}",
                va='center', fontsize=8.2, fontweight='bold', color='#212121')

# 2. Right Panel: Donut Chart showing Proportionate Compute Footprint
wedges, texts, autotexts = ax_pie.pie(
    latencies,
    labels=[s['id'] for s in stages_info],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.78,
    textprops=dict(fontsize=8.5, fontweight='bold'),
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2.0)
)
for at in autotexts:
    at.set_fontsize(7.8)
    at.set_color('white')
    at.set_weight('bold')

ax_pie.text(0, 0, f"TOTAL\n{total_time:.2f}s", ha='center', va='center', fontsize=11, fontweight='bold', color='#1a237e')
ax_pie.set_title('Pipeline Latency Distribution (%)', fontsize=11.5, fontweight='bold', pad=10)

bottleneck = max(stages_info, key=lambda s: s['latency'])
banner_text = f"PRIMARY BOTTLENECK: {bottleneck['id']} ({bottleneck['name']}) -> {bottleneck['latency']:.2f}s ({bottleneck['pct']:.1f}% of total)\n" \
              f"Hardware: GPU 0 (Qwen2.5-VL / SigLIP) + GPU 1 (SAM3) + CPU (Spatial Entropy Partitioning)"
fig.text(0.5, 0.02, banner_text, ha='center', fontsize=8.5, style='italic', color='#37474f',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f5', edgecolor='#b0bec5', linewidth=1.2))

plt.tight_layout(rect=[0, 0.08, 1, 0.96])
plt.show()

# 3. Formatted Benchmark Summary Table
print('=' * 105)
print(f"{'Stage':<9} | {'Component Name':<34} | {'Device':<24} | {'Time (s)':<10} | {'% Total':<8} | {'Candidate Funnel'}")
print('-' * 105)
for s in stages_info:
    print(f"{s['id']:<9} | {s['name']:<34} | {s['device']:<24} | {s['latency']:>7.3f}s  | {s['pct']:>6.1f}% | {s['funnel']}")
print('=' * 105)
print(f"TOTAL PIPELINE EXECUTION TIME: {total_time:.3f}s (100.0%)")
print(f"PRIMARY SYSTEM BOTTLENECK:    {bottleneck['id']} ({bottleneck['name']}) at {bottleneck['latency']:.3f}s ({bottleneck['pct']:.1f}%)")
print('=' * 105)
